In [33]:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    # Nothing to do in a local environment (VSCode/JupyterLab)
    pass

In [34]:
from dataclasses import dataclass, field, asdict
from typing import List, Literal, Optional, Dict, Any, Callable
from abc import ABC, abstractmethod
import json, math, numpy as np
from enum import Enum


# ========== Type Definitions ==========
class SpendingFunction(str, Enum):
    OBRIEN_FLEMING = "obrien_fleming"
    POCOCK = "pocock"
    HSD = "hsd"


class TestType(str, Enum):
    TWO_PROPORTIONS = "two_sample_proportions"
    TWO_MEANS = "two_sample_means"
    TIME_TO_EVENT = "time_to_event"


class TimingType(str, Enum):
    INFORMATION_BASED = "information_based"
    CALENDAR_TIME = "calendar_time"


class InformationSpacing(str, Enum):
    EQUAL = "equal"
    CUSTOM = "custom"


# ========== Hierarchical Configuration Components ==========


@dataclass
class TestConfig:
    """Test-specific configuration"""

    test_type: TestType = TestType.TWO_PROPORTIONS
    sided: Literal["one", "two"] = "one"
    alpha: float = 0.025
    power: float = 0.90


@dataclass
class SequentialConfig:
    """Sequential design configuration"""

    n_analyses: int = 3
    timing_type: TimingType = TimingType.INFORMATION_BASED
    info_spacing: InformationSpacing = InformationSpacing.EQUAL
    info_times: Optional[List[float]] = None

    def get_info_times(self, n_analyses: int) -> np.ndarray:
        """Get information times (custom or equally spaced)"""
        if self.info_times and self.info_spacing == InformationSpacing.CUSTOM:
            t = np.array(self.info_times, dtype=float)
        else:
            t = np.array([(i + 1) / n_analyses for i in range(n_analyses)], dtype=float)

        # Ensure monotonic increase and within (0, 1]
        t = np.clip(t, 1e-6, 1.0)
        t = np.maximum.accumulate(t)
        return t


@dataclass
class AllocationConfig:
    """Sample allocation configuration"""

    alloc_ratio: float = 1.0  # nB / nA (treatment / control)


@dataclass
class BoundaryConfig:
    """Efficacy and futility boundary configuration"""

    # Efficacy boundary
    spending_function: SpendingFunction = SpendingFunction.OBRIEN_FLEMING
    hsd_gamma: float = -4.0

    # Futility boundary
    futility_enabled: bool = False
    futility_spending: Optional[SpendingFunction] = None
    futility_z: Optional[float] = None


@dataclass
class SimulationConfig:
    """Simulation parameters"""

    seed: int = 42
    n_sims: int = 5000


@dataclass
class DisplayConfig:
    """Display and formatting configuration"""

    digits: int = 4  # Significant figures
    ddigits: int = 2  # Decimal places
    tdigits: int = 1  # Time display precision


# ========== Effect Specification (test-type specific) ==========


@dataclass
class ProportionsEffect:
    """Effect specification for two-sample proportions test"""

    p_control: float = 0.10
    p_treatment: Optional[float] = None
    effect_size: Optional[float] = 0.02
    effect_type: Literal["absolute", "relative", "odds_ratio"] = "absolute"

    def get_treatment_proportion(self) -> float:
        """Calculate treatment proportion based on effect specification"""
        if self.p_treatment is not None:
            return self.p_treatment

        if self.effect_type == "absolute":
            return self.p_control + self.effect_size
        elif self.effect_type == "relative":
            return self.p_control * (1 + self.effect_size)
        elif self.effect_type == "odds_ratio":
            odds_control = self.p_control / (1 - self.p_control)
            odds_treatment = odds_control * self.effect_size
            return odds_treatment / (1 + odds_treatment)
        else:
            raise ValueError(f"Unknown effect_type: {self.effect_type}")


@dataclass
class TimeToEventEffect:
    """Effect specification for time-to-event analysis"""

    hazard_ratio: float = 0.6


@dataclass
class MeansEffect:
    """Effect specification for two-sample means test"""

    mean_control: float = 10.0
    mean_treatment: Optional[float] = None
    effect_size: Optional[float] = 2.0
    std_dev: float = 5.0
    pooled_std: bool = True


# ========== Sample Size Specification (test-type specific) ==========


@dataclass
class ProportionsSampleSize:
    """Sample size specification for proportions test"""

    n_per_analysis: int = 500  # Per group


@dataclass
class TimeToEventSampleSize:
    """Sample size specification for time-to-event analysis"""

    total_events: int = 171
    total_sample_size: int = 296
    time_unit: Literal["days", "weeks", "months", "years"] = "months"
    accrual_duration: Optional[float] = None
    follow_up_duration: Optional[float] = None


@dataclass
class MeansSampleSize:
    """Sample size specification for means test"""

    n_per_analysis: int = 100  # Per group


# ========== Effect Calculator ABC ==========


class EffectCalculator(ABC):
    """Abstract base class for effect size calculations"""

    @abstractmethod
    def standardized_effect(self, spec: "DesignSpec", info_time: float) -> float:
        """Calculate standardized effect at given information time"""
        pass

    @abstractmethod
    def sample_sizes(self, spec: "DesignSpec") -> Dict[str, np.ndarray]:
        """Calculate sample sizes at each analysis"""
        pass


# ========== Main Design Specifications ==========


@dataclass
class DesignSpec:
    """
    Hierarchical design specification for sequential trials

    Organized hierarchical structure:
    - test: Test configuration (type, significance level, power)
    - sequential: Sequential design configuration (# analyses, information times)
    - allocation: Sample allocation configuration
    - boundary: Boundary configuration (efficacy and futility)
    - simulation: Simulation configuration
    - display: Display configuration
    - effect: Effect size specification (test-type specific)
    - sample_size: Sample size specification (test-type specific)
    """

    test: TestConfig = field(default_factory=TestConfig)
    sequential: SequentialConfig = field(default_factory=SequentialConfig)
    allocation: AllocationConfig = field(default_factory=AllocationConfig)
    boundary: BoundaryConfig = field(default_factory=BoundaryConfig)
    simulation: SimulationConfig = field(default_factory=SimulationConfig)
    display: DisplayConfig = field(default_factory=DisplayConfig)

    # Test-type specific configurations (set based on test_type)
    effect: Any = None
    sample_size: Any = None

    def resolved_info_times(self) -> np.ndarray:
        """Get resolved information times"""
        return self.sequential.get_info_times(self.sequential.n_analyses)

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for serialization"""
        d = asdict(self)

        # Convert Enums to strings
        def convert_enums(obj):
            if isinstance(obj, dict):
                return {k: convert_enums(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_enums(item) for item in obj]
            elif isinstance(obj, Enum):
                return obj.value
            else:
                return obj

        return convert_enums(d)

    def to_json(self, indent: int = 2) -> str:
        """Export as JSON string"""
        return json.dumps(self.to_dict(), ensure_ascii=False, indent=indent)


# ========== Concrete Design Specifications ==========


@dataclass
class ProportionsDesignSpec(DesignSpec):
    """Design specification for two-sample proportions test"""

    effect: ProportionsEffect = field(default_factory=ProportionsEffect)
    sample_size: ProportionsSampleSize = field(default_factory=ProportionsSampleSize)

    def __post_init__(self):
        self.test.test_type = TestType.TWO_PROPORTIONS


@dataclass
class TimeToEventDesignSpec(DesignSpec):
    """Design specification for time-to-event analysis"""

    effect: TimeToEventEffect = field(default_factory=TimeToEventEffect)
    sample_size: TimeToEventSampleSize = field(default_factory=TimeToEventSampleSize)

    def __post_init__(self):
        self.test.test_type = TestType.TIME_TO_EVENT


@dataclass
class MeansDesignSpec(DesignSpec):
    """Design specification for two-sample means test"""

    effect: MeansEffect = field(default_factory=MeansEffect)
    sample_size: MeansSampleSize = field(default_factory=MeansSampleSize)

    def __post_init__(self):
        self.test.test_type = TestType.TWO_MEANS

In [35]:
from scipy.stats import norm
import pandas as pd


# ========== Boundary Value Calculation ==========
class BoundaryCalculator:
    """Calculate critical boundary values for sequential designs"""

    @staticmethod
    def spending_function(
        func: SpendingFunction, t: np.ndarray, alpha: float, gamma: float = -4.0
    ) -> np.ndarray:
        """Calculate cumulative alpha spending at information times"""
        if func == SpendingFunction.OBRIEN_FLEMING:
            # Lan-DeMets O'Brien-Fleming approximation
            z_alpha = norm.ppf(1 - alpha)
            return 2 * (1 - norm.cdf(z_alpha / np.sqrt(t)))
        elif func == SpendingFunction.POCOCK:
            # Pocock-like spending
            return alpha * np.log(1 + (np.e - 1) * t)
        elif func == SpendingFunction.HSD:
            # Hwang-Shih-DeCani
            if abs(gamma) < 1e-12:
                return alpha * t
            return alpha * (1 - np.exp(-gamma * t)) / (1 - np.exp(-gamma))
        else:
            raise ValueError(f"Unknown spending function: {func}")

    @staticmethod
    def critical_values(spec: DesignSpec) -> Dict[str, Any]:
        """Calculate critical values for efficacy and futility bounds"""
        t = spec.resolved_info_times()
        k = len(t)

        # Efficacy spending
        alpha_total = (
            spec.test.alpha if spec.test.sided == "one" else spec.test.alpha / 2.0
        )
        cumulative_alpha = BoundaryCalculator.spending_function(
            spec.boundary.spending_function, t, alpha_total, spec.boundary.hsd_gamma
        )

        # Incremental alpha at each analysis
        alpha_increments = np.diff(np.concatenate([[0.0], cumulative_alpha]))

        # Critical Z-values (approximate, assuming independence)
        z_efficacy = norm.ppf(1 - alpha_increments)

        # Futility bounds (if enabled)
        z_futility = None
        if spec.boundary.futility_enabled:
            if spec.boundary.futility_z is not None:
                # Non-binding constant futility bound
                z_futility = np.full(k, spec.boundary.futility_z)
            elif spec.boundary.futility_spending is not None:
                # Spending function based futility
                beta = 1 - spec.test.power
                cumulative_beta = BoundaryCalculator.spending_function(
                    spec.boundary.futility_spending, t, beta, spec.boundary.hsd_gamma
                )
                beta_increments = np.diff(np.concatenate([[0.0], cumulative_beta]))
                z_futility = norm.ppf(beta_increments)
            else:
                # Binding futility proportional to efficacy (if not specified otherwise)
                z_futility = 0.5 * z_efficacy

        return {
            "info_times": t,
            "cumulative_alpha": cumulative_alpha,
            "z_efficacy": z_efficacy,
            "z_futility": z_futility,
        }


# ========== Effect Size and Sample Size Calculation ==========
class ProportionsEffectCalculator(EffectCalculator):
    """Effect size calculator for two proportions"""

    def standardized_effect(
        self, spec: ProportionsDesignSpec, info_time: float
    ) -> float:
        """Calculate standardized effect (delta) for proportions test"""
        p_A = spec.effect.p_control
        p_B = spec.effect.get_treatment_proportion()

        # Pooled proportion under H0
        p_pooled = (p_A + p_B) / 2.0

        # Sample sizes at this information time
        n_total_A = spec.sample_size.n_per_analysis * spec.sequential.n_analyses
        n_A = int(n_total_A * info_time)
        n_B = int(n_A * spec.allocation.alloc_ratio)

        if n_A == 0 or n_B == 0:
            return 0.0

        # Standard error
        se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_A + 1 / n_B))

        if se == 0:
            return 0.0

        # Standardized difference
        return (p_B - p_A) / se

    def sample_sizes(self, spec: ProportionsDesignSpec) -> Dict[str, np.ndarray]:
        """Calculate sample sizes at each analysis"""
        t = spec.resolved_info_times()
        n_total_A = spec.sample_size.n_per_analysis * spec.sequential.n_analyses

        n_A = (n_total_A * t).astype(int)
        n_B = (n_A * spec.allocation.alloc_ratio).astype(int)

        return {
            "n_control": n_A,
            "n_treatment": n_B,
            "n_total": n_A + n_B,
            "info_fraction": t,
        }


class TimeToEventEffectCalculator(EffectCalculator):
    """Effect size calculator for time-to-event"""

    def standardized_effect(
        self, spec: TimeToEventDesignSpec, info_time: float
    ) -> float:
        """Calculate standardized effect for survival analysis"""
        # For log-rank test, effect is based on events
        hr = spec.effect.hazard_ratio
        log_hr = np.log(hr)

        # Events at this information time
        events = int(spec.sample_size.total_events * info_time)

        if events == 0:
            return 0.0

        # Standard error of log(HR) ~ sqrt(4/events) for equal allocation
        r = spec.allocation.alloc_ratio
        se = np.sqrt((1 + r) ** 2 / (r * events))

        return log_hr / se

    def sample_sizes(self, spec: TimeToEventDesignSpec) -> Dict[str, np.ndarray]:
        """Calculate events and sample sizes at each analysis"""
        t = spec.resolved_info_times()

        events = (spec.sample_size.total_events * t).astype(int)
        n_total = spec.sample_size.total_sample_size
        n_A = int(n_total / (1 + spec.allocation.alloc_ratio))
        n_B = n_total - n_A

        return {
            "events": events,
            "n_control": np.full_like(events, n_A),
            "n_treatment": np.full_like(events, n_B),
            "n_total": np.full_like(events, n_total),
            "info_fraction": t,
        }


class MeansEffectCalculator(EffectCalculator):
    """Effect size calculator for two means"""

    def standardized_effect(self, spec: MeansDesignSpec, info_time: float) -> float:
        """Calculate standardized effect for means test"""
        mu_A = spec.effect.mean_control
        mu_B = (
            spec.effect.mean_treatment
            if spec.effect.mean_treatment is not None
            else mu_A + spec.effect.effect_size
        )
        sigma = spec.effect.std_dev

        # Sample sizes at this information time
        n_total_A = spec.sample_size.n_per_analysis * spec.sequential.n_analyses
        n_A = int(n_total_A * info_time)
        n_B = int(n_A * spec.allocation.alloc_ratio)

        if n_A == 0 or n_B == 0:
            return 0.0

        # Standard error
        se = sigma * np.sqrt(1 / n_A + 1 / n_B)

        if se == 0:
            return 0.0

        return (mu_B - mu_A) / se

    def sample_sizes(self, spec: MeansDesignSpec) -> Dict[str, np.ndarray]:
        """Calculate sample sizes at each analysis"""
        t = spec.resolved_info_times()
        n_total_A = spec.sample_size.n_per_analysis * spec.sequential.n_analyses

        n_A = (n_total_A * t).astype(int)
        n_B = (n_A * spec.allocation.alloc_ratio).astype(int)

        return {
            "n_control": n_A,
            "n_treatment": n_B,
            "n_total": n_A + n_B,
            "info_fraction": t,
        }

In [36]:
# ========== Simulation and Power Computation ==========
class SimulationEngine:
    """Run simulations to estimate power and operating characteristics"""

    @staticmethod
    def simulate_trial(
        spec: DesignSpec,
        boundaries: Dict[str, Any],
        effect_calc: EffectCalculator,
        rng: np.random.Generator,
    ) -> Dict[str, Any]:
        """Simulate a single trial"""
        t = boundaries["info_times"]
        z_upper = boundaries["z_efficacy"]
        z_lower = boundaries["z_futility"]

        k = len(t)
        stopped = False
        stop_analysis = None
        stop_reason = None

        # Generate Z-statistics trajectory
        Z = np.zeros(k)
        for i in range(k):
            if stopped:
                Z[i] = Z[i - 1]  # Carry forward
            else:
                # Drift under alternative
                drift = effect_calc.standardized_effect(spec, t[i])
                # Brownian motion increment
                if i == 0:
                    Z[i] = rng.normal(drift * np.sqrt(t[i]), np.sqrt(t[i]))
                else:
                    dt = t[i] - t[i - 1]
                    Z[i] = Z[i - 1] + rng.normal(drift * np.sqrt(dt), np.sqrt(dt))

                # Check boundaries
                if Z[i] >= z_upper[i]:
                    stopped = True
                    stop_analysis = i + 1
                    stop_reason = "efficacy"
                elif z_lower is not None and Z[i] <= z_lower[i]:
                    stopped = True
                    stop_analysis = i + 1
                    stop_reason = "futility"

        if not stopped:
            stop_analysis = k
            stop_reason = "final"

        return {
            "Z": Z,
            "stopped_at": stop_analysis,
            "reason": stop_reason,
            "reject_h0": stop_reason == "efficacy",
        }

    @staticmethod
    def run_simulations(
        spec: DesignSpec, boundaries: Dict[str, Any], effect_calc: EffectCalculator
    ) -> Dict[str, Any]:
        """Run full simulation study"""
        rng = np.random.default_rng(spec.simulation.seed)

        results = []
        for _ in range(spec.simulation.n_sims):
            results.append(
                SimulationEngine.simulate_trial(spec, boundaries, effect_calc, rng)
            )

        # Aggregate results
        rejections = sum(r["reject_h0"] for r in results)
        power = rejections / spec.simulation.n_sims

        # Stop distribution
        stop_dist = {}
        for r in results:
            a = r["stopped_at"]
            stop_dist[a] = stop_dist.get(a, 0) + 1

        # Expected sample size
        sample_info = effect_calc.sample_sizes(spec)
        if "n_total" in sample_info:
            n_per_analysis = sample_info["n_total"]
        elif "events" in sample_info:
            n_per_analysis = sample_info["events"]

        ess = sum(
            stop_dist.get(i + 1, 0) * n_per_analysis[i]
            for i in range(len(boundaries["info_times"]))
        )
        ess /= spec.simulation.n_sims

        return {
            "power": power,
            "expected_sample_size": ess,
            "stop_distribution": stop_dist,
            "max_sample_size": n_per_analysis[-1],
            "results": results,
        }

In [37]:
# ========== Design Lab: Main Orchestrator ==========
class DesignLab:
    """
    Main orchestrator for sequential design planning

    Provides a unified interface for:
    - Configuring sequential designs
    - Computing boundaries
    - Running simulations
    - Generating reports
    """

    def __init__(self, spec: DesignSpec):
        self.spec = spec
        self.boundaries = None
        self.simulation_results = None
        self._effect_calculator = self._create_effect_calculator()

    def _create_effect_calculator(self) -> EffectCalculator:
        """Factory method to create appropriate effect calculator"""
        if isinstance(self.spec, ProportionsDesignSpec):
            return ProportionsEffectCalculator()
        elif isinstance(self.spec, TimeToEventDesignSpec):
            return TimeToEventEffectCalculator()
        elif isinstance(self.spec, MeansDesignSpec):
            return MeansEffectCalculator()
        else:
            raise ValueError(f"Unsupported design spec type: {type(self.spec)}")

    def compute_boundaries(self) -> "DesignLab":
        """Compute critical boundaries"""
        self.boundaries = BoundaryCalculator.critical_values(self.spec)
        return self

    def run_simulations(self) -> "DesignLab":
        """Run simulation study"""
        if self.boundaries is None:
            self.compute_boundaries()

        self.simulation_results = SimulationEngine.run_simulations(
            self.spec, self.boundaries, self._effect_calculator
        )
        return self

    def get_summary(self) -> pd.DataFrame:
        """Get summary table of design characteristics"""
        if self.boundaries is None:
            self.compute_boundaries()

        t = self.boundaries["info_times"]
        z_upper = self.boundaries["z_efficacy"]
        z_lower = self.boundaries["z_futility"]
        alpha_cum = self.boundaries["cumulative_alpha"]

        # Sample size information
        sample_info = self._effect_calculator.sample_sizes(self.spec)

        data = {
            "Analysis": np.arange(1, len(t) + 1),
            "Info Fraction": t,
            "Z Efficacy": z_upper,
            "Cumulative α": alpha_cum,
        }

        # Add sample size columns
        if "n_control" in sample_info:
            data["N Control"] = sample_info["n_control"]
            data["N Treatment"] = sample_info["n_treatment"]
            data["N Total"] = sample_info["n_total"]
        elif "events" in sample_info:
            data["Events"] = sample_info["events"]
            data["N Control"] = sample_info["n_control"]
            data["N Treatment"] = sample_info["n_treatment"]

        if z_lower is not None:
            data["Z Futility"] = z_lower

        df = pd.DataFrame(data)

        # Format numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if "Info" in col or "α" in col:
                df[col] = df[col].round(self.spec.display.ddigits + 2)
            elif col.startswith("Z"):
                df[col] = df[col].round(self.spec.display.ddigits)
            else:
                df[col] = df[col].round(0).astype(int)

        return df

    def get_power_summary(self) -> Dict[str, Any]:
        """Get power and operating characteristics summary"""
        if self.simulation_results is None:
            self.run_simulations()

        return {
            "Estimated Power": f"{self.simulation_results['power']:.1%}",
            "Expected Sample Size": f"{self.simulation_results['expected_sample_size']:.0f}",
            "Maximum Sample Size": f"{self.simulation_results['max_sample_size']:.0f}",
            "Efficiency (ESS/Max)": f"{self.simulation_results['expected_sample_size'] / self.simulation_results['max_sample_size']:.1%}",
        }

    def plot_boundaries(
        self, show_trajectories: bool = False, n_trajectories: int = 20
    ):
        """Plot critical boundaries and optionally sample trajectories"""
        import matplotlib.pyplot as plt

        if self.boundaries is None:
            self.compute_boundaries()

        t = self.boundaries["info_times"]
        z_upper = self.boundaries["z_efficacy"]
        z_lower = self.boundaries["z_futility"]

        fig, ax = plt.subplots(figsize=(10, 6))

        # Efficacy boundary
        ax.plot(t, z_upper, "r-", linewidth=2, label="Efficacy Boundary", marker="o")

        # Futility boundary
        if z_lower is not None:
            ax.plot(
                t, z_lower, "b-", linewidth=2, label="Futility Boundary", marker="s"
            )

        # Sample trajectories
        if show_trajectories and self.simulation_results is not None:
            for i in range(
                min(n_trajectories, len(self.simulation_results["results"]))
            ):
                Z = self.simulation_results["results"][i]["Z"]
                ax.plot(t, Z, "gray", alpha=0.3, linewidth=0.5)

        ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.set_xlabel("Information Fraction", fontsize=12)
        ax.set_ylabel("Z-statistic", fontsize=12)
        ax.set_title(
            f'{self.spec.boundary.spending_function.value.replace("_", " ").title()} Boundaries',
            fontsize=14,
        )
        ax.legend()
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        return fig

    def to_dict(self) -> Dict[str, Any]:
        """Export full design information as dictionary"""
        result = {
            "specification": self.spec.to_dict(),
        }

        if self.boundaries is not None:
            result["boundaries"] = {
                k: v.tolist() if isinstance(v, np.ndarray) else v
                for k, v in self.boundaries.items()
            }

        if self.simulation_results is not None:
            sim_copy = self.simulation_results.copy()
            sim_copy.pop("results", None)  # Don't include all individual results
            result["simulation_summary"] = sim_copy

        return result

    def export_json(self, filepath: Optional[str] = None) -> str:
        """Export design to JSON file or return JSON string"""
        json_str = json.dumps(self.to_dict(), ensure_ascii=False, indent=2)
        if filepath:
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(json_str)
            return f"Design exported to {filepath}"
        return json_str

    def __repr__(self) -> str:
        lines = [
            f"DesignLab({self.spec.test.test_type.value})",
            f"  Analyses: {self.spec.sequential.n_analyses}",
            f"  Alpha: {self.spec.test.alpha}",
            f"  Spending: {self.spec.boundary.spending_function.value}",
        ]

        if self.simulation_results:
            lines.append(f"  Power: {self.simulation_results['power']:.1%}")

        return "\n".join(lines)

In [38]:
from enum import Enum
from scipy.optimize import minimize_scalar, minimize
from typing import Protocol

# ========== Design Mode Framework ==========


class DesignMode(str, Enum):
    """Design modes representing different use case patterns"""

    FIXED_TIMING = "fixed_timing"  # Fixed timing, compute boundaries
    OPTIMIZE_ASN = "optimize_asn"  # Fixed max N, optimize ASN
    OPTIMIZE_DESIGN = "optimize_design"  # Optimize alpha, power, effect size, etc.
    FIXED_POWER = "fixed_power"  # Fixed power, compute required sample size


class DesignObjective(ABC):
    """Abstract base class representing design objectives"""

    @abstractmethod
    def evaluate(self, spec: DesignSpec, lab: DesignLab) -> float:
        """Evaluate objective function (return value to minimize)"""
        pass

    @abstractmethod
    def get_constraints(self, spec: DesignSpec) -> Dict[str, Any]:
        """Return constraint conditions"""
        pass


class MinimizeASN(DesignObjective):
    """Minimize ASN (Average Sample Number / Expected sample size)"""

    def __init__(self, max_n: int, target_power: float = 0.90):
        self.max_n = max_n
        self.target_power = target_power

    def evaluate(self, spec: DesignSpec, lab: DesignLab) -> float:
        """Compute ASN (with penalty for constraint violations)"""
        try:
            lab.compute_boundaries()
            lab.run_simulations()

            # Check power constraint
            power = lab.simulation_results["power"]
            if power < self.target_power:
                # Penalty for insufficient power
                return self.max_n * 10 + (self.target_power - power) * 10000

            # Check maximum sample size constraint
            max_sample = lab.simulation_results["max_sample_size"]
            if max_sample > self.max_n:
                return self.max_n * 10 + (max_sample - self.max_n) * 100

            # Return ASN
            return lab.simulation_results["expected_sample_size"]
        except:
            return self.max_n * 100

    def get_constraints(self, spec: DesignSpec) -> Dict[str, Any]:
        return {"max_n": self.max_n, "target_power": self.target_power}


class MaximizePower(DesignObjective):
    """Maximize power with fixed sample size"""

    def __init__(self, max_n: int):
        self.max_n = max_n

    def evaluate(self, spec: DesignSpec, lab: DesignLab) -> float:
        """Compute power (return negative value to convert maximization to minimization)"""
        try:
            lab.compute_boundaries()
            lab.run_simulations()

            max_sample = lab.simulation_results["max_sample_size"]
            if max_sample > self.max_n:
                return 1.0  # Penalty

            power = lab.simulation_results["power"]
            return -power  # Negative for minimization
        except:
            return 1.0

    def get_constraints(self, spec: DesignSpec) -> Dict[str, Any]:
        return {"max_n": self.max_n}


class BalancedDesign(DesignObjective):
    """Optimize balance between power and sample size"""

    def __init__(
        self,
        power_weight: float = 1.0,
        sample_weight: float = 1.0,
        target_power: float = 0.90,
        target_n: int = 3000,
    ):
        self.power_weight = power_weight
        self.sample_weight = sample_weight
        self.target_power = target_power
        self.target_n = target_n

    def evaluate(self, spec: DesignSpec, lab: DesignLab) -> float:
        """Composite objective function: weighted score of power and sample size"""
        try:
            lab.compute_boundaries()
            lab.run_simulations()

            power = lab.simulation_results["power"]
            ess = lab.simulation_results["expected_sample_size"]

            # Normalized scores
            power_score = abs(power - self.target_power) / self.target_power
            sample_score = abs(ess - self.target_n) / self.target_n

            return self.power_weight * power_score + self.sample_weight * sample_score
        except:
            return 100.0

    def get_constraints(self, spec: DesignSpec) -> Dict[str, Any]:
        return {
            "target_power": self.target_power,
            "target_n": self.target_n,
            "power_weight": self.power_weight,
            "sample_weight": self.sample_weight,
        }


# ========== Design Optimizer ==========


class DesignOptimizer:
    """Design optimization engine"""

    def __init__(self, base_spec: DesignSpec, objective: DesignObjective):
        self.base_spec = base_spec
        self.objective = objective
        self.optimization_history = []

    def optimize_info_times(self, n_analyses: int) -> np.ndarray:
        """Optimize information times"""

        def objective_func(x):
            """Objective function to optimize"""
            # x is [t1, t2, ..., t_{k-1}] (last one fixed at 1.0)
            info_times = np.append(x, 1.0)
            info_times = np.sort(info_times)  # Ensure monotonic increase

            # Update configuration
            spec = self._clone_spec()
            spec.sequential.info_times = info_times.tolist()
            spec.sequential.info_spacing = InformationSpacing.CUSTOM

            # Evaluate
            lab = DesignLab(spec)
            score = self.objective.evaluate(spec, lab)

            self.optimization_history.append(
                {"info_times": info_times.tolist(), "score": score}
            )

            return score

        # Initial guess: equal spacing
        x0 = np.array([(i + 1) / n_analyses for i in range(n_analyses - 1)])

        # Constraints: 0 < t1 < t2 < ... < t_{k-1} < 1
        bounds = [(0.1, 0.99) for _ in range(n_analyses - 1)]

        # Execute optimization
        result = minimize(
            objective_func,
            x0,
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": 50},
        )

        optimal_times = np.append(result.x, 1.0)
        return np.sort(optimal_times)

    def optimize_n_analyses(self, min_k: int = 2, max_k: int = 10) -> int:
        """Search for optimal number of analyses"""

        best_k = min_k
        best_score = float("inf")

        for k in range(min_k, max_k + 1):
            spec = self._clone_spec()
            spec.sequential.n_analyses = k

            lab = DesignLab(spec)
            score = self.objective.evaluate(spec, lab)

            if score < best_score:
                best_score = score
                best_k = k

        return best_k

    def optimize_alpha(self, min_alpha: float = 0.001, max_alpha: float = 0.1) -> float:
        """Search for optimal alpha level"""

        def objective_func(alpha):
            spec = self._clone_spec()
            spec.test.alpha = alpha
            lab = DesignLab(spec)
            return self.objective.evaluate(spec, lab)

        result = minimize_scalar(
            objective_func, bounds=(min_alpha, max_alpha), method="bounded"
        )

        return result.x

    def optimize_comprehensive(self) -> DesignSpec:
        """Comprehensive optimization"""
        spec = self._clone_spec()

        # Step 1: Optimize number of analyses
        optimal_k = self.optimize_n_analyses()
        spec.sequential.n_analyses = optimal_k

        # Step 2: Optimize information times
        optimal_times = self.optimize_info_times(optimal_k)
        spec.sequential.info_times = optimal_times.tolist()
        spec.sequential.info_spacing = InformationSpacing.CUSTOM

        return spec

    def _clone_spec(self) -> DesignSpec:
        """Create a clone of the design specification"""
        import copy

        return copy.deepcopy(self.base_spec)

    def get_optimization_summary(self) -> pd.DataFrame:
        """Return optimization history as DataFrame"""
        if not self.optimization_history:
            return pd.DataFrame()

        return pd.DataFrame(self.optimization_history)

In [41]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt

# ========== Mode-Based Modular UI ==========


class DesignLabUIModular:
    """
    Modular, mode-based design planner UI

    Supports multiple use case patterns:
    - Fixed Timing: timing is fixed, compute boundaries
    - Optimize ASN: max N is fixed, optimize ASN
    - Optimize Design: search for optimal combination of α, power, effect size, etc.
    - Fixed Power: power is fixed, compute required sample size
    """

    _singleton_instance = None  # For singleton enforcement

    def __new__(cls, *args, **kwargs):
        # Ensure only one instance exists in the notebook
        if cls._singleton_instance is not None:
            return cls._singleton_instance
        instance = super().__new__(cls)
        cls._singleton_instance = instance
        return instance

    def __init__(self, initial_spec: Optional[DesignSpec] = None):
        if hasattr(self, "_initialized") and self._initialized:
            return
        self._initialized = True

        self.spec = initial_spec or ProportionsDesignSpec()
        self.lab = DesignLab(self.spec)
        self.optimizer = None
        self.current_mode = DesignMode.FIXED_TIMING

        # Initialize widgets
        self._create_widgets()
        self._create_layout()
        self._attach_observers()

        # Initial calculation
        self._update_design()

    def _create_widgets(self):
        """Create widgets"""

        # ========== Mode Selection ==========
        self.w_mode = widgets.Dropdown(
            options=[
                ("Fixed Timing → Compute Boundaries", DesignMode.FIXED_TIMING.value),
                ("Fixed Max N → Optimize ASN", DesignMode.OPTIMIZE_ASN.value),
                (
                    "Flexible Parameters → Optimize Design",
                    DesignMode.OPTIMIZE_DESIGN.value,
                ),
                ("Fixed Power → Compute Required N", DesignMode.FIXED_POWER.value),
            ],
            value=DesignMode.FIXED_TIMING.value,
            description="Design Mode:",
            style={"description_width": "120px"},
            layout=widgets.Layout(width="600px"),
        )

        # ========== Common Parameters ==========
        self.w_alpha = widgets.FloatText(
            value=0.025,
            min=0.001,
            max=0.5,
            step=0.001,
            description="Alpha (α):",
            style={"description_width": "120px"},
        )

        self.w_power = widgets.FloatText(
            value=0.90,
            min=0.50,
            max=0.99,
            step=0.01,
            description="Power (1-β):",
            style={"description_width": "120px"},
        )

        self.w_n_analyses = widgets.IntText(
            value=3,
            min=2,
            max=20,
            step=1,
            description="# Analyses:",
            style={"description_width": "120px"},
        )

        self.w_spending_func = widgets.Dropdown(
            options=[
                ("O'Brien-Fleming", "obrien_fleming"),
                ("Pocock", "pocock"),
                ("HSD", "hsd"),
            ],
            value="obrien_fleming",
            description="Spending Fn:",
            style={"description_width": "120px"},
        )

        # ========== Effect Parameters ==========
        self.w_p_control = widgets.FloatText(
            value=0.10,
            min=0.001,
            max=0.999,
            step=0.001,
            description="p (Control):",
            style={"description_width": "120px"},
        )

        self.w_effect_size = widgets.FloatText(
            value=0.02,
            min=0.001,
            max=1.0,
            step=0.001,
            description="Effect Size:",
            style={"description_width": "120px"},
        )

        # ========== Mode-Specific Parameters ==========

        # Fixed Timing mode
        self.w_info_times_text = widgets.Textarea(
            value="0.33, 0.67, 1.0",
            description="Info Times:",
            style={"description_width": "120px"},
            layout=widgets.Layout(width="400px", height="60px"),
        )

        self.w_n_per_analysis = widgets.IntText(
            value=500,
            min=10,
            max=10000,
            description="N per Analysis:",
            style={"description_width": "120px"},
        )

        # Optimize ASN mode
        self.w_max_n_total = widgets.IntText(
            value=3000,
            min=100,
            max=20000,
            description="Max N Total:",
            style={"description_width": "120px"},
        )

        self.w_target_power = widgets.FloatText(
            value=0.90,
            min=0.50,
            max=0.99,
            step=0.01,
            description="Target Power:",
            style={"description_width": "120px"},
        )

        # Optimize Design mode
        self.w_alpha_range_min = widgets.FloatText(
            value=0.01,
            min=0.001,
            max=0.5,
            step=0.001,
            description="Alpha Min:",
            style={"description_width": "120px"},
        )

        self.w_alpha_range_max = widgets.FloatText(
            value=0.05,
            min=0.001,
            max=0.5,
            step=0.001,
            description="Alpha Max:",
            style={"description_width": "120px"},
        )

        self.w_k_range_min = widgets.IntText(
            value=2,
            min=1,
            max=20,
            step=1,
            description="K Min:",
            style={"description_width": "120px"},
        )

        self.w_k_range_max = widgets.IntText(
            value=6,
            min=1,
            max=20,
            step=1,
            description="K Max:",
            style={"description_width": "120px"},
        )

        self.w_optimize_criterion = widgets.Dropdown(
            options=[
                ("Minimize ASN", "minimize_asn"),
                ("Maximize Power", "maximize_power"),
                ("Balanced", "balanced"),
            ],
            value="minimize_asn",
            description="Criterion:",
            style={"description_width": "120px"},
        )

        # Action Buttons
        self.w_compute_btn = widgets.Button(
            description="🔧 Compute Design",
            button_style="primary",
            tooltip="Compute boundaries for fixed timing",
        )

        self.w_optimize_btn = widgets.Button(
            description="🎯 Run Optimization",
            button_style="success",
            tooltip="Run design optimization",
        )

        self.w_simulate_btn = widgets.Button(
            description="🎲 Run Simulation",
            button_style="info",
            tooltip="Run power simulation",
        )

        # Output areas
        self.output_config = widgets.Output()
        self.output_summary = widgets.Output()
        self.output_plot = widgets.Output()
        self.output_power = widgets.Output()
        self.output_optimization = widgets.Output()

        # Event handlers
        self.w_mode.observe(self._on_mode_change, "value")
        self.w_compute_btn.on_click(self._on_compute_click)
        self.w_optimize_btn.on_click(self._on_optimize_click)
        self.w_simulate_btn.on_click(self._on_simulate_click)

    def _create_layout(self):
        """Create layout (top: config, bottom: results)"""

        # === Top: Design Configuration ===

        # Mode selection (always shown)
        mode_box = widgets.VBox(
            [widgets.HTML("<h3>📋 Design Mode</h3>"), self.w_mode, widgets.HTML("<hr>")]
        )

        # Common parameters
        common_box = widgets.VBox(
            [
                widgets.HTML("<h4>Common Parameters</h4>"),
                self.w_alpha,
                self.w_power,
                self.w_n_analyses,
                self.w_spending_func,
                self.w_p_control,
                self.w_effect_size,
            ]
        )

        # Mode-specific parameters (switch dynamically)
        self.mode_specific_box = widgets.VBox(
            [
                widgets.HTML("<h4>Mode-Specific Parameters</h4>"),
                self.w_info_times_text,
                self.w_n_per_analysis,
            ]
        )

        # Action buttons
        actions_box = widgets.HBox(
            [self.w_compute_btn, self.w_optimize_btn, self.w_simulate_btn],
            layout=widgets.Layout(justify_content="space-around"),
        )

        # Configuration panel (upper part)
        config_panel = widgets.VBox(
            [
                mode_box,
                widgets.HBox(
                    [common_box, self.mode_specific_box],
                    layout=widgets.Layout(justify_content="space-between"),
                ),
                widgets.HTML("<hr>"),
                actions_box,
            ],
            layout=widgets.Layout(
                padding="15px",
                border="2px solid #ddd",
                border_radius="10px",
                background_color="#f9f9f9",
            ),
        )

        # === Bottom: Design Results ===

        # Results tabs (singleton)
        if not hasattr(self, "results_tabs"):
            self.results_tabs = widgets.Tab()
            self.results_tabs.children = [
                self.output_summary,
                self.output_plot,
                self.output_power,
                self.output_optimization,
            ]
            self.results_tabs.set_title(0, "📊 Summary")
            self.results_tabs.set_title(1, "📈 Boundaries")
            self.results_tabs.set_title(2, "⚡ Power")
            self.results_tabs.set_title(3, "🎯 Optimization")
        else:
            # Always ensure only one set of children
            self.results_tabs.children = [
                self.output_summary,
                self.output_plot,
                self.output_power,
                self.output_optimization,
            ]

        results_panel = widgets.VBox(
            [widgets.HTML("<h2>📊 Design Results</h2>"), self.results_tabs],
            layout=widgets.Layout(
                padding="15px",
                border="2px solid #ddd",
                border_radius="10px",
                margin_top="20px",
            ),
        )

        # Main layout (vertical: config on top, results on bottom)
        self.main_layout = widgets.VBox(
            [
                widgets.HTML("<h1>🎛️ Design Configuration</h1>"),
                config_panel,
                results_panel,
            ]
        )

    def _attach_observers(self):
        """Set event handlers (lightweight)"""
        # Only mode switching triggers update; others are manual
        pass

    def _on_mode_change(self, change):
        """Handler for mode switching"""
        mode = DesignMode(change["new"])
        self.current_mode = mode

        # Update mode-specific UI
        if mode == DesignMode.FIXED_TIMING:
            self.mode_specific_box.children = [
                widgets.HTML("<h4>Fixed Timing Parameters</h4>"),
                self.w_info_times_text,
                self.w_n_per_analysis,
                widgets.HTML(
                    "<p><i>Specify information times and compute boundaries</i></p>"
                ),
            ]
            self.w_compute_btn.disabled = False
            self.w_optimize_btn.disabled = True

        elif mode == DesignMode.OPTIMIZE_ASN:
            self.mode_specific_box.children = [
                widgets.HTML("<h4>ASN Optimization Parameters</h4>"),
                self.w_max_n_total,
                self.w_target_power,
                widgets.HTML(
                    "<p><i>Fix max N and optimize information times for min ASN</i></p>"
                ),
            ]
            self.w_compute_btn.disabled = True
            self.w_optimize_btn.disabled = False

        elif mode == DesignMode.OPTIMIZE_DESIGN:
            self.mode_specific_box.children = [
                widgets.HTML("<h4>Design Optimization Parameters</h4>"),
                widgets.HBox([self.w_alpha_range_min, self.w_alpha_range_max]),
                widgets.HBox([self.w_k_range_min, self.w_k_range_max]),
                self.w_optimize_criterion,
                widgets.HTML(
                    "<p><i>Find optimal combination of alpha, K, and timing</i></p>"
                ),
            ]
            self.w_compute_btn.disabled = True
            self.w_optimize_btn.disabled = False

        elif mode == DesignMode.FIXED_POWER:
            self.mode_specific_box.children = [
                widgets.HTML("<h4>Fixed Power Parameters</h4>"),
                self.w_target_power,
                widgets.HTML(
                    "<p><i>Fix power and compute required sample size</i></p>"
                ),
            ]
            self.w_compute_btn.disabled = False
            self.w_optimize_btn.disabled = True

    def _on_compute_click(self, button):
        """Handler for Compute button click"""
        self._update_spec_from_widgets()

        with self.output_summary:
            clear_output(wait=True)
            display(HTML("<h3>⏳ Computing...</h3>"))

        try:
            self.lab = DesignLab(self.spec)
            self.lab.compute_boundaries()
            self._display_results()
        except Exception as e:
            with self.output_summary:
                clear_output(wait=True)
                display(HTML(f"<p style='color: red;'><b>Error:</b> {str(e)}</p>"))

    def _on_optimize_click(self, button):
        """Handler for Optimize button click"""
        self._update_spec_from_widgets()

        with self.output_optimization:
            clear_output(wait=True)
            display(HTML("<h3>🎯 Running Optimization...</h3>"))

        try:
            if self.current_mode == DesignMode.OPTIMIZE_ASN:
                objective = MinimizeASN(
                    max_n=self.w_max_n_total.value,
                    target_power=self.w_target_power.value,
                )
                self.optimizer = DesignOptimizer(self.spec, objective)

                # Optimize information times
                optimal_times = self.optimizer.optimize_info_times(
                    self.spec.sequential.n_analyses
                )

                # Update settings
                self.spec.sequential.info_times = optimal_times.tolist()
                self.spec.sequential.info_spacing = InformationSpacing.CUSTOM

                # Calculate results
                self.lab = DesignLab(self.spec)
                self.lab.compute_boundaries()
                self.lab.run_simulations()

                self._display_optimization_results(optimal_times)
                self._display_results()

            elif self.current_mode == DesignMode.OPTIMIZE_DESIGN:
                # Comprehensive optimization
                criterion = self.w_optimize_criterion.value

                if criterion == "minimize_asn":
                    objective = MinimizeASN(
                        max_n=10000, target_power=self.w_power.value
                    )
                elif criterion == "maximize_power":
                    objective = MaximizePower(max_n=10000)
                else:
                    objective = BalancedDesign()

                self.optimizer = DesignOptimizer(self.spec, objective)
                optimized_spec = self.optimizer.optimize_comprehensive()

                self.spec = optimized_spec
                self.lab = DesignLab(self.spec)
                self.lab.compute_boundaries()
                self.lab.run_simulations()

                self._display_optimization_results(self.spec.sequential.info_times)
                self._display_results()

        except Exception as e:
            with self.output_optimization:
                clear_output(wait=True)
                display(
                    HTML(
                        f"<p style='color: red;'><b>Optimization Error:</b> {str(e)}</p>"
                    )
                )

    def _on_simulate_click(self, button):
        """Handler for Simulate button click"""
        try:
            if self.lab.boundaries is None:
                self.lab.compute_boundaries()

            with self.output_power:
                clear_output(wait=True)
                display(HTML("<h3>🎲 Running Simulation...</h3>"))

            self.lab.run_simulations()
            self._display_power_results()

        except Exception as e:
            with self.output_power:
                clear_output(wait=True)
                display(
                    HTML(
                        f"<p style='color: red;'><b>Simulation Error:</b> {str(e)}</p>"
                    )
                )

    def _update_spec_from_widgets(self):
        """Update settings from widgets"""
        # Common parameters
        self.spec.test.alpha = self.w_alpha.value
        self.spec.test.power = self.w_power.value
        self.spec.sequential.n_analyses = self.w_n_analyses.value
        self.spec.boundary.spending_function = SpendingFunction(
            self.w_spending_func.value
        )

        # Effect parameters
        if isinstance(self.spec, ProportionsDesignSpec):
            self.spec.effect.p_control = self.w_p_control.value
            self.spec.effect.effect_size = self.w_effect_size.value

        # Mode-specific
        if self.current_mode == DesignMode.FIXED_TIMING:
            # Parse info times
            try:
                times_str = self.w_info_times_text.value
                times = [float(t.strip()) for t in times_str.split(",")]
                self.spec.sequential.info_times = times
                self.spec.sequential.info_spacing = InformationSpacing.CUSTOM
            except:
                self.spec.sequential.info_spacing = InformationSpacing.EQUAL

            if isinstance(self.spec, ProportionsDesignSpec):
                self.spec.sample_size.n_per_analysis = self.w_n_per_analysis.value

    def _display_results(self):
        """Display results"""
        # Summary
        with self.output_summary:
            clear_output(wait=True)
            display(HTML("<h3>Design Summary</h3>"))
            display(self.lab.get_summary())

        # Plot
        with self.output_plot:
            clear_output(wait=True)
            fig = self.lab.plot_boundaries()
            plt.show()

    def _display_power_results(self):
        """Display power analysis results"""
        with self.output_power:
            clear_output(wait=True)
            display(HTML("<h3>Power Analysis Results</h3>"))
            power_summary = self.lab.get_power_summary()

            html = "<table style='width:100%; border-collapse: collapse; margin-top: 10px;'>"
            for key, value in power_summary.items():
                html += f"<tr><td style='padding: 8px; border: 1px solid #ddd; background-color: #f0f0f0;'><b>{key}</b></td>"
                html += f"<td style='padding: 8px; border: 1px solid #ddd;'>{value}</td></tr>"
            html += "</table>"
            display(HTML(html))

    def _display_optimization_results(self, optimal_times):
        """Display optimization results"""
        with self.output_optimization:
            clear_output(wait=True)
            display(HTML("<h3>Optimization Results</h3>"))

            display(
                HTML(
                    f"<p><b>Optimal Information Times:</b> {', '.join([f'{t:.3f}' for t in optimal_times])}</p>"
                )
            )

            if self.optimizer and self.optimizer.optimization_history:
                history_df = self.optimizer.get_optimization_summary()
                display(HTML("<h4>Optimization History (Last 10 iterations)</h4>"))
                display(history_df.tail(10))

    def _update_design(self):
        """Initial design calculation"""
        self._on_compute_click(None)

    def display(self):
        """Display UI, ensuring only one instance is shown and previous widgets are closed."""
        from IPython.display import display, clear_output

        if (
            hasattr(self, "_last_displayed_widget")
            and self._last_displayed_widget is not self.main_layout
        ):
            try:
                self._last_displayed_widget.close()
            except Exception:
                pass
        self._last_displayed_widget = self.main_layout
        clear_output(wait=True)
        display(self.main_layout)

In [ ]:
ui_modular = DesignLabUIModular()
ui_modular.display()